# Record of the number of cards on [jp web](https://www.pokemon-card.com/card-search/index.php)

2020, 7, 25: 12141 件

In [1]:
import CardScraperJP
import warnings
warnings.filterwarnings('ignore')

# SM: start: 33001, end: 38493
# XY: start:, end: 32388
#     start:, end: 28915
start = 30000
end   = 38494
CardScraperJP.scrapeCards(start, end)

[====================] 100%	(8495/8495)

In [2]:
import CardScraperJP
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Add new block to scrape new cards
# Do NOT override successful history!!
# 2020.08.08
start = 38494
end   = 38499
CardScraperJP.scrapeCards(start, end)

[====================] 100%	(6/6)

In [2]:
import bs4
import requests


In [3]:
def readEnergy(p):
    # <span class="icon-psychic icon">
    spans = p.find_all('span')
    if spans:
        energies = [span['class'][0].split('-')[1] for span in spans]
        for i in range(len(energies)):
            p = str(p).replace(str(spans[i]), energies[i])
        p = bs4.BeautifulSoup(p)
    p = p.get_text().replace('\n   ', '')
    return p

In [4]:
def readPrismstar(h1):
    # <span class="pcg pcg-prismstar">
    span = h1.find('span')
    if span:
        prismstar = span['class'][1].split('-')[1]
        h1 = str(h1).replace(str(span), prismstar)
        h1 = bs4.BeautifulSoup(h1)
    h1 = h1.get_text().replace('\n   ', '')
    return h1

In [5]:
def readMega(p):
    # <span class="pcg pcg-megamark"></span>
    span = p.find('span')
    if span:
        mega = span['class'][1].split('-')[1]
        p = str(h1).replace(str(span), mega)
        p = bs4.BeautifulSoup(h1)
    p = h1.get_text().replace('\n   ', '')
    return p

In [6]:
def readEnergyMegaPrismstar(p):
    # <span class="pcg pcg-megamark"></span>
    # <span class="pcg pcg-prismstar">
    # <span class="icon-psychic icon">
    spans = p.find_all('span')
    marks = []
    if spans:
        for span in spans:
            if 'icon' in str(span):
                marks.append(span['class'][0].split('-')[1])
            elif 'mega' in str(span):
                marks.append(span['class'][1].split('-')[1][:4])
            elif 'prismstar' in str(span):
                marks.append(span['class'][1].split('-')[1])
        
        for i in range(len(marks)):
            p = str(p).replace(str(spans[i]), marks[i])
        p = bs4.BeautifulSoup(p)
    p = p.get_text().replace('\n   ', '')
    return p

In [7]:
def getContent(cardId):
    # anti-scraping
    user_agent = "Mozilla/5.0 (Windows NT 10.0; WOW64; rv:68.0) Gecko/20100101 Firefox/68.0"
    url = f'https://www.pokemon-card.com/card-search/details.php/card/{cardId}'
    response = requests.get(url, headers={'User-Agent': user_agent})
    if response.status_code == 200:
        # print(response.content.decode('utf-8'))
        return response.content.decode('utf-8')
    else:
        print(f"Fail to get the url [{response.status_code}]")
        return [cardId, response.status_code]

In [8]:
def readCard(content):
    # start reading content
    soup = bs4.BeautifulSoup(content, 'html.parser')
    card = soup.section
    # all type cards
    name = readEnergyMegaPrismstar(card.h1)
    img = card.find('img', class_='fit')['src']
    
    # init
    [reg,         setNum,    setCount,   rarity,      dexNum,    dexClass,
     height,      weight,    dexDesc,    author,      desc,      stage, 
     hp,          pType,     ability,    abilityDesc, waza1Cost, waza1Name,
     waza1Damage, waza1Desc, waza2Cost,  waza2Name,   waza2Damage, waza2Desc,
     GXCost, GXName, GXDamage, GXDesc,
     weakType,    weakValue, resistType, resistValue, escape, spRule] = [''] * 34
    
    # decide card type
    cardType = card.h2.get_text()
    if cardType == '基本エネルギー':
        return [cardType, name, img, reg, setNum, setCount, rarity, dexNum,
            dexClass, height, weight, dexDesc, author, desc, stage, hp, pType,
            ability, abilityDesc, waza1Cost, waza1Name, waza1Damage,
            waza1Desc, waza2Cost, waza2Name, waza2Damage, waza2Desc,
            GXCost, GXName, GXDamage, GXDesc,
            weakType, weakValue, resistType, resistValue,
            escape, spRule]


    ## left box
    reg = card.find('img', class_='img-regulation')['alt'] # regulation
    regImg = card.find('img', class_='img-regulation')['src']
    
    setInfo = card.find('div', class_='subtext').get_text().strip()
    if len(setInfo.split('/')) == 2:
        setCount = setInfo.strip()[-3:]
        if setCount.isdigit():
            setCount = int(setCount)
        setNum = int(setInfo.strip()[:3])
    else:
        setCount = setInfo

    if card.find('img', width='24'):
        rarityImg = card.find('img', width='24')['src']
        rarity = rarityImg.split('.')[0].split('ic_')[1]
    
    if cardType == '特殊エネルギー':
        desc = readEnergyMegaPrismstar(card.find('p'))
        return [cardType, name, img, reg, setNum, setCount, rarity, dexNum,
            dexClass, height, weight, dexDesc, author, desc, stage, hp, pType,
            ability, abilityDesc, waza1Cost, waza1Name, waza1Damage,
            waza1Desc, waza2Cost, waza2Name, waza2Damage, waza2Desc,
                GXCost, GXName, GXDamage, GXDesc,
            weakType, weakValue, resistType, resistValue,
            escape, spRule]
    
    author = card.find('div', class_='author').get_text().strip()
    if author:
        author = card.find('div', class_='author').get_text().strip().split('\n')[1]

    ### national pokedex
    pokedex = card.find('div', class_='card')
    if pokedex: # has pokedex
        if pokedex.h4:
            dexline = pokedex.h4.get_text().strip().split('\u3000')
            if len(dexline) == 2:
                [dexNum, dexClass] = dexline
                dexNum = int(dexNum.split('.')[1])
            elif len(dexline) == 1:
                if any(char.isdigit() for char in dexline[0]):
                    dexNum = dexline[0]
                else:
                    dexClass = dexline[0]
        if len(pokedex.find_all('p')) == 2:
            htAndWt = pokedex.p.get_text().split('：')
            height = float(htAndWt[1].split(' ')[0])
            weight = float(htAndWt[2].split(' ')[0])
            dexDesc = pokedex.find_all('p')[1].get_text()
        elif len(pokedex.find_all('p')) == 1 and '重さ' in pokedex.find('p').get_text():
            htAndWt = pokedex.p.get_text().split('：')
            height = float(htAndWt[1].split(' ')[0])
            weight = float(htAndWt[2].split(' ')[0])
        elif len(pokedex.find_all('p')) == 1:
            dexDesc = pokedex.find('p').get_text()
        else:
            print('Something wrong with the pokedex!')
    
    if cardType in ['サポート', 'グッズ', 'ポケモンのどうぐ', 'スタジアム']:
        desc = card.find_all('p')
        if cardType in ['サポート', 'グッズ', 'スタジアム']:
            desc = readEnergyMegaPrismstar(desc[0])
        if cardType == 'ポケモンのどうぐ':
            desc = readEnergyMegaPrismstar(desc[1])
        return [cardType, name, img, reg, setNum, setCount, rarity, dexNum,
            dexClass, height, weight, dexDesc, author, desc, stage, hp, pType,
            ability, abilityDesc, waza1Cost, waza1Name, waza1Damage,
            waza1Desc, waza2Cost, waza2Name, waza2Damage, waza2Desc,
                GXCost, GXName, GXDamage, GXDesc,
            weakType, weakValue, resistType, resistValue,
            escape, spRule]
    
    cardType = 'pokemon'

    ## right box
    stage = card.find('span', class_='type').get_text()
    if '\xa0' in stage:
        stage = stage.replace('\xa0', ' ')
    hp = card.find('span', class_='hp-num').get_text()
    hp = int(hp)
    topSpans = card.find('div', class_='td-r').find_all('span')
    topSpansClass = [span['class'] for span in topSpans]
    pTypes = [s for s in topSpansClass if 'icon' in s]
    pType = [l[0].split('-')[1] for l in pTypes]

    ### waza part
    part = content.split('<span class="hp-type">タイプ</span>')[1].split('</table>')[0].strip()
    soup = bs4.BeautifulSoup(part)
    wazaPart = bs4.BeautifulSoup(soup.prettify(formatter="minimal"))
    
    h2 = wazaPart.find_all('h2')
    skills = wazaPart.find_all('h4')
    if not skills[-1].get_text().strip():
        # empty (wrong special rule as void)
        del skills[-1]
    p = wazaPart.find_all('p')
    if not skills[0].get_text().strip():
        # mega evolution rule (delete)
        del skills[0]
        del p[0]
    
    for area in h2:
        areaType = area.get_text().strip()
        if areaType in ["特性", "古代能力"]:
            # ability or ancient trait
            # print('learning an ability')
            ability = skills[0].get_text().strip()
            abilityDesc = readEnergyMegaPrismstar(p[0]).strip()
            del skills[0]

        elif areaType == "特別なルール":
            # special rule
            # print('learning a special rule')
            spRule = readEnergyMegaPrismstar(p[-1]).strip()
            del p[-1]
            
        elif areaType == "GXワザ":
            # GX waza
            [GXCost, GXName, GXDamage, GXDesc] = [''] * 4
            # print('learning a GX attack')
            GXCost = [span['class'][0].split('-')[1] for span in skills[-1].find_all('span', class_='icon')]
            GX = skills[-1].get_text().strip().split(' ')
            GXName = GX[0].strip()
            GXDamage = GX[-1]
            if not GXDamage[-2].isdigit():
                GXDamage = ''
            GXDesc = skills[-1].find_next_sibling('p')
            GXDesc = readEnergyMegaPrismstar(GXDesc).strip()
            del skills[-1], p[-2]
            
        elif areaType == "ワザ":
            # waza
            waza1Cost = [span['class'][0].split('-')[1] for span in skills[0].find_all('span', class_='icon')]
            waza1 = skills[0].get_text().strip().split(' ')
            waza1Name = waza1[0].strip()
            waza1Damage = waza1[-1]
            if not waza1Damage[-2].isdigit():
                waza1Damage = ''
            waza1Desc = skills[0].find_next_sibling('p')
            waza1Desc = readEnergyMegaPrismstar(waza1Desc).strip()

            if len(skills) > 1:
                waza2Cost = [span['class'][0].split('-')[1] for span in skills[1].find_all('span', class_='icon')]
                waza2 = skills[1].get_text().strip().split(' ')
                waza2Name = waza2[0].strip()
                waza2Damage = waza2[-1]
                if not waza2Damage[-2].isdigit():
                    waza2Damage = ''
                waza2Desc = skills[1].find_next_sibling('p')
                waza2Desc = readEnergyMegaPrismstar(waza2Desc).strip()
                
        else:
            print(f"{name} has an unseen areaType: {areaType}!!")


    ### table
    td = wazaPart.find_all('td')
    if td[0].find('span'):
        weakType = td[0].find('span')['class'][0].split('-')[1]
        weakValue = td[0].get_text().strip()

    if td[1].find('span'):
        resistType = td[1].find('span')['class'][0].split('-')[1]
        resistValue = td[1].get_text().strip()

    escape = len(td[2].find_all('span'))
    

    return [cardType, name, img, reg, setNum, setCount, rarity, dexNum,
            dexClass, height, weight, dexDesc, author, desc, stage, hp, pType,
            ability, abilityDesc, waza1Cost, waza1Name, waza1Damage,
            waza1Desc, waza2Cost, waza2Name, waza2Damage, waza2Desc,
            GXCost, GXName, GXDamage, GXDesc,
            weakType, weakValue, resistType, resistValue,
            escape, spRule]

In [9]:
# read pokemon card
# url = "https://www.pokemon-card.com/card-search/details.php/card/38407/regu/XY" # biidoru c
# url = "https://www.pokemon-card.com/card-search/details.php/card/38439/regu/XY" # zacian as
# url = "https://www.pokemon-card.com/card-search/details.php/card/38438/regu/XY" # mahoippu Vmax
# url = "https://www.pokemon-card.com/card-search/details.php/card/38426/regu/XY" # zekuromu r
# url = "https://www.pokemon-card.com/card-search/details.php/card/38419/regu/XY" # zaruudo V 38493
# url = "https://www.pokemon-card.com/card-search/details.php/card/38423/regu/XY" # raiboruto c ability
# url = "https://www.pokemon-card.com/card-search/details.php/card/38310/regu/XY" # mew V resistance
# url = "https://www.pokemon-card.com/card-search/details.php/card/38300/regu/XY" # pikachu V
# url = "https://www.pokemon-card.com/card-search/details.php/card/38304/regu/XY" # denryu


cardId = 37409 # start: 33001, end: 38493
# url = f'https://www.pokemon-card.com/card-search/details.php/card/{cardId}'
# content = getContent(url)
readCard(getContent(cardId))

Something wrong with the pokedex!


['pokemon',
 'レックウザEX',
 '/assets/images/card_images/large/XY/037409_P_REKKUUZAEX.jpg',
 'XY',
 19,
 48,
 '',
 'No.384',
 '',
 '',
 '',
 '',
 'Ryota Murayama',
 '',
 'たね',
 170,
 ['none'],
 '',
 '',
 ['none'],
 'ライジングバーン',
 '10＋',
 '相手のバトルポケモンが「ポケモンEX」なら、50ダメージを追加。',
 ['none', 'none', 'none'],
 'りゅうのはどう',
 '100',
 '自分の山札を上から3枚トラッシュする。',
 '',
 '',
 '',
 '',
 'electric',
 '×2',
 'fighting',
 '',
 2,
 '']

In [50]:
# read other card
cardId = 33640 # magearna 801
# cardId = 34000 # hikaru
# cardId = 35898 # GX
# cardId = 37281 # prismstar
# cardId = 38474 # dougu ポケモンのどうぐ
# cardId = 38470 # guzzu グッズ
# cardId = 33300 # guzzu S-P
# cardId = 33207 # stadium
# cardId = 37281
# cardId = 38477 # supporter サポート
# cardId = 38480 # special energy 特殊エネルギー
# cardId = 37743 # basic energy 基本エネルギー
# cardId = 37392 # EX
cardId = 37406 # Mega (double type)
# cardId = 33580 # break (no waza)
# cardId = 33592 # read mega icon
# cardId = 32388 # out of range
# cardId = 36742 # no illustrator
# cardId = 37214 # web wrong


# url = f'https://www.pokemon-card.com/card-search/details.php/card/{cardId}'
content = getContent(cardId)

soup = bs4.BeautifulSoup(content, 'html.parser')
card = soup.section
# all type cards
name = readEnergyMegaPrismstar(card.h1)
img = card.find('img', class_='fit')['src']

In [51]:
card

<section class="Section">
<h1 class="Heading1 mt20"><span class="pcg pcg-megamark"></span>サーナイトEX</h1>
<div class="Box">
<div class="LeftBox">
<img alt="メガサーナイトEX" class="fit" src="/assets/images/card_images/large/XY/037406_P_MSANAITOEX.jpg"/>
<div class="subtext Text-fjalla">
<img alt="XY" class="img-regulation" src="/assets/images/card/regulation_logo_1/XY.gif"/> 016 / 048                     </div>
<div class="card Text-fjalla">
<h4>No.282　</h4> <p>高さ：1.6 m　　重さ：48.4 kg</p> </div>
<div class="author">
<h4>イラストレーター</h4>
<a href="/card-search/index.php?regulation_illust=all&amp;illust=5ban Graphics">5ban Graphics</a> </div>
</div>
<div class="RightBox">
<div class="RightBox-inner">
<div class="TopInfo Text-fjalla">
<div class="tr">
<div class="td-l">
<span class="type">M進化</span>
</div>
<div class="td-r">
<span class="hp">HP</span>
<span class="hp-num">210</span>
<span class="hp-type">タイプ</span>
<span class="icon-fairy icon"></span><span class="icon-psychic icon"></span> </div>
</div>


# Test double types

In [52]:
pType = card.find('div', class_='td-r').find_all('span')[-1]['class'][0].split('-')[1]
pType

'psychic'

In [53]:
topSpans = card.find('div', class_='td-r').find_all('span')
topSpansClass = [span['class'] for span in topSpans]
pTypes = [s for s in topSpansClass if 'icon' in s]
pType = [l[0].split('-')[1] for l in pTypes]
pType

['fairy', 'psychic']

# Test Illustrator

In [54]:
card.find('div', class_='author').get_text().strip()

'イラストレーター\n5ban Graphics'

# Test Pokédex

In [55]:
pokedex = card.find('div', class_='card')
pokedex

<div class="card Text-fjalla">
<h4>No.282　</h4> <p>高さ：1.6 m　　重さ：48.4 kg</p> </div>

In [56]:
pokedex.h4.get_text().strip().split('\u3000')

['No.282']

In [57]:
[dexNum, dexClass] = pokedex.h4.get_text().strip().split('\u3000')
dexNum = int(dexNum.split('.')[1])
htAndWt = pokedex.p.get_text().split('：')
height = float(htAndWt[1].split(' ')[0])
weight = float(htAndWt[2].split(' ')[0])
dexDesc = pokedex.find_all('p')[1].get_text()

ValueError: not enough values to unpack (expected 2, got 1)

# Test `span` inside `p` tag

In [58]:
[span['class'][1].split('-')[1] for span in card.h1.find_all('span')]

['megamark']

In [59]:
p = card.find('p')
p

<p>高さ：1.6 m　　重さ：48.4 kg</p>

In [60]:
p.get_text()

'高さ：1.6 m\u3000\u3000重さ：48.4 kg'

In [61]:
readEnergy(p)

'高さ：1.6 m\u3000\u3000重さ：48.4 kg'

In [62]:
spans = p.find_all('span')
spans

[]

In [63]:
energies = [span['class'][0].split('-')[1] for span in spans]
energies

[]

In [64]:

for i in range(len(energies)):
    p = str(p).replace(str(spans[0]), energies[0])
# p = bs4.BeautifulSoup(p)
# p = p.get_text().replace('\n   ', '')

p

<p>高さ：1.6 m　　重さ：48.4 kg</p>

In [65]:
# decide card type
cardType = card.h2.get_text()
if cardType == '基本エネルギー':
    return
elif cardType in ['特殊エネルギー', 'サポート', 'グッズ', 'ポケモンのどうぐ']:
    desc = card.find_all('p')
else:
    cardType = 'pokemon'

SyntaxError: 'return' outside function (<ipython-input-65-af2deb471edd>, line 4)

# Test waza part

In [66]:
part = content.split('<span class="hp-type">タイプ</span>')[1].split('</table>')[0].strip()
soup = bs4.BeautifulSoup(part)
wazaPart = bs4.BeautifulSoup(soup.prettify(formatter="minimal"))

In [67]:

wazaPart

<html>
<body>
<span class="icon-fairy icon">
</span>
<span class="icon-psychic icon">
</span>
<h2 class="mt20">
   ワザ
  </h2>
<h4>
<span class="icon-void icon">
</span>
</h4>
<p>
   【M進化】ポケモンになったとき、自分の番は終わる。
  </p>
<h4>
<span class="icon-fairy icon">
</span>
<span class="icon-none icon">
</span>
   ディスペアーレイ
  </h4>
<p>
   自分のベンチポケモンを好きなだけトラッシュし、トラッシュしたベンチポケモンの数×10ダメージを追加。
  </p>
<h2 class="mt20">
   特別なルール
  </h2>
<p>
   ポケモンEXがきぜつしたとき、相手はサイドを2枚とる。
  </p>
<table cellpadding="0" cellspacing="0">
<tr>
<th>
     弱点
    </th>
<th>
     抵抗力
    </th>
<th>
     にげる
    </th>
</tr>
<tr>
<td>
<span class="icon-steel icon">
</span>
     ×2
    </td>
<td>
<span class="icon-dark icon">
</span>
</td>
<td class="escape">
<span class="icon-none icon">
</span>
<span class="icon-none icon">
</span>
</td>
</tr>
</table>
</body>
</html>

In [68]:
h2 = wazaPart.find_all('h2')
h2

[<h2 class="mt20">
    ワザ
   </h2>,
 <h2 class="mt20">
    特別なルール
   </h2>]

In [69]:
h2 = wazaPart.find_all('h2')
skills = wazaPart.find_all('h4')
skills

[<h4>
 <span class="icon-void icon">
 </span>
 </h4>,
 <h4>
 <span class="icon-fairy icon">
 </span>
 <span class="icon-none icon">
 </span>
    ディスペアーレイ
   </h4>]

In [70]:
h2 = wazaPart.find_all('h2')
skills = wazaPart.find_all('h4')
p = wazaPart.find_all('p')
p

[<p>
    【M進化】ポケモンになったとき、自分の番は終わる。
   </p>,
 <p>
    自分のベンチポケモンを好きなだけトラッシュし、トラッシュしたベンチポケモンの数×10ダメージを追加。
   </p>,
 <p>
    ポケモンEXがきぜつしたとき、相手はサイドを2枚とる。
   </p>]

In [78]:
wazaCount = len(p) - len(h2) +1

h2 = wazaPart.find_all('h2')
skills = wazaPart.find_all('h4')
if not skills[-1].get_text().strip():
    # empty (wrong special rule as void)
    del skills[-1]
p = wazaPart.find_all('p')
if not skills[0].get_text().strip():
    # mega evolution rule (delete)
    del skills[0]
    del p[0]

# init
[ability, abilityDesc, spRule] = [''] * 3 
for area in h2:
    areaType = area.get_text().strip()
    if areaType in ["特性", "古代能力"]:
        # ability or ancient trait
        print('learning an ability')
        ability = skills[0].get_text().strip()
        abilityDesc = p[0].get_text().strip()
        del skills[0]
        
    elif areaType == "特別なルール":
        # special rule
        print('learning a special rule')
        spRule = p[-1].get_text().strip()
        del p[-1]
        
    elif areaType == "GXワザ":
        # GX waza
        [GXCost, GXName, GXDamage, GXDesc] = [''] * 4
        print('learning a GX attack')
        GXCost = [span['class'][0].split('-')[1] for span in skills[-1].find_all('span', class_='icon')]
        GX = skills[-1].get_text().strip().split(' ')
        GXName = GX[0].strip()
        GXDamage = GX[-1]
        if not GXDamage[-2].isdigit():
            GXDamage = ''
        GXDesc = skills[-1].find_next_sibling('p')
        GXDesc = readEnergy(GXDesc).strip()
        del skills[-1], p[-2]
        
[wazaCount, skills, p]
# [GXCost, GXName, GXDamage, GXDesc]

learning a special rule


[1,
 [<h4>
  <span class="icon-fairy icon">
  </span>
  <span class="icon-none icon">
  </span>
     ディスペアーレイ
    </h4>],
 [<p>
     自分のベンチポケモンを好きなだけトラッシュし、トラッシュしたベンチポケモンの数×10ダメージを追加。
    </p>]]

In [79]:
skills


[<h4>
 <span class="icon-fairy icon">
 </span>
 <span class="icon-none icon">
 </span>
    ディスペアーレイ
   </h4>]

In [80]:
# init
[waza1Cost, waza1Name, waza1Damage, waza1Desc] = [''] * 4
[waza2Cost, waza2Name, waza2Damage, waza2Desc] = [''] * 4

waza1Cost = [span['class'][0].split('-')[1] for span in skills[0].find_all('span', class_='icon')]
waza1 = skills[0].get_text().strip().split(' ')
waza1Name = waza1[0].strip()
waza1Damage = waza1[-1]
if not waza1Damage[-2].isdigit():
    waza1Damage = ''
waza1Desc = skills[0].find_next_sibling('p')
waza1Desc = readEnergyMegaPrismstar(waza1Desc).strip()

if len(skills) > 1:
    waza2Cost = [span['class'][0].split('-')[1] for span in skills[1].find_all('span', class_='icon')]
    waza2 = skills[1].get_text().strip().split(' ')
    waza2Name = waza2[0].strip()
    waza2Damage = waza2[-1]
    if not waza2Damage[-2].isdigit():
        waza2Damage = ''
    waza2Desc = skills[1].find_next_sibling('p')
    waza2Desc = readEnergyMegaPrismstar(waza2Desc).strip()

In [81]:
waza1 = skills[0].get_text().strip().split(' ')
waza1Name = waza1[0].strip()
waza1Name

'ディスペアーレイ'

In [75]:
name

'megaサーナイトEX'

In [468]:
spRule

''

In [469]:
td = wazaPart.find_all('td')
td

[<td>
 <span class="icon-fire icon">
 </span>
      ×2
     </td>,
 <td>
      --
     </td>,
 <td class="escape">
 <span class="icon-none icon">
 </span>
 </td>]

In [472]:
[weakType, weakValue, resistType, resistValue] = [''] * 4

if td[0].find('span'):
    weakType = td[0].find('span')['class'][0].split('-')[1]
    weakValue = td[0].get_text().strip()

if td[1].find('span'):
    resistType = td[1].find('span')['class'][0].split('-')[1]
    resistValue = td[1].get_text().strip()
    
escape = len(td[2].find_all('span'))

[weakType, weakValue, resistType, resistValue, escape]

['fire', '×2', '', '', 1]

In [471]:
td[0].find('span')['class'][0].split('-')[1]

'fire'

In [56]:
# special engergy
twinEngergy = "https://www.pokemon-card.com/card-search/details.php/card/38482/regu/all"
card = readCard(twinEngergy)
card

<section class="Section">
<h1 class="Heading1 mt20">ツインエネルギー</h1>
<div class="Box">
<div class="LeftBox">
<img alt="ツインエネルギー" class="fit" src="/assets/images/card_images/large/S3a/038482_E_TSUINENERUGI.jpg"/>
<div class="subtext Text-fjalla">
<img alt="S3a" class="img-regulation" src="/assets/images/card/regulation_logo_1/S3a.gif"/> 076 / 076 <img src="/assets/images/card/rarity/ic_rare_u_c.gif" width="24"/> </div>
<span> </span>
<div class="author">
</div>
</div>
<div class="RightBox">
<div class="RightBox-inner">
<h2 class="mt20">特殊エネルギー</h2>
<p>このカードは、ポケモンについているかぎり、【無】エネルギー2個ぶんとしてはたらく。<br/>
<br/>
「ポケモンV・GX」についているなら、【無】エネルギー1個ぶんとしてはたらく。</p>
</div>
</div>
<div class="clear"></div>
</div>
</section>

In [57]:
# supporter
onion = "https://www.pokemon-card.com/card-search/details.php/card/38477/regu/XY"
card = readCard(onion)
card

<section class="Section">
<h1 class="Heading1 mt20">オニオン</h1>
<div class="Box">
<div class="LeftBox">
<img alt="オニオン" class="fit" src="/assets/images/card_images/large/S3a/038477_T_ONION.jpg"/>
<div class="subtext Text-fjalla">
<img alt="S3a" class="img-regulation" src="/assets/images/card/regulation_logo_1/S3a.gif"/> 071 / 076 <img src="/assets/images/card/rarity/ic_rare_u_c.gif" width="24"/> </div>
<span> </span>
<div class="author">
<h4>イラストレーター</h4>
<a href="/card-search/index.php?regulation_illust=XY&amp;illust=take">take</a> </div>
</div>
<div class="RightBox">
<div class="RightBox-inner">
<h2 class="mt20">サポート</h2>
<p>自分の山札を3枚引く。その後、自分の手札を3枚まで選び、トラッシュする。（必ず1枚はトラッシュする。）</p>
<p>サポートは、自分の番に1枚しか使えない。</p>
</div>
</div>
<div class="clear"></div>
</div>
</section>

In [ ]:
cardId = 38477 # start: 33001, end: 38493
readCard(cardId)

In [22]:
import sys
import pandas as pd

start = 38491
end = 38493

def scrapeCards(start, end):
    columns = ['cardId', 'cardType', 'name', 'img', 'regulation', 'setNum', 'setCount', 'rarity', 'dexNum',
                'dexClass', 'height', 'weight', 'dexDesc', 'author', 'desc', 'stage', 'hp', 'pType',
                'ability', 'abilityDesc', 'waza1Cost', 'waza1Name', 'waza1Damage',
                'waza1Desc', 'waza2Cost', 'waza2Name', 'waza2Damage', 'waza2Desc',
                'GXCost', 'GXName', 'GXDamage', 'GXDesc',
                'weakType', 'weakValue', 'resistType', 'resistValue', 'escape', 'spRule']

    cardDF = pd.DataFrame(columns=columns)
    errorDF = pd.DataFrame(columns=['errorCardId'])
    
    n = end - start+1
    for i in range(n):
        cardId = start + i
        content = getContent(cardId)
        soup = bs4.BeautifulSoup(content, 'html.parser')
        if soup.section:
            cardDF.loc[i] = readCard(content)
        else:
            errorDF.loc[i] = cardId
        j = (i + 1) / n
        sys.stdout.write('\r')
        # the exact output you're looking for:
        sys.stdout.write("[%-20s] %d%%" % ('='*int(20*j), 100*j))
        sys.stdout.flush()

    cardDF.reset_index().to_csv(f'cards_jp_{start}_{end}.csv')
    if len(errorDF) > 0:
        errorDF.reset_index().to_csv(f'error_id_{start}_{end}.csv')

In [23]:
scrapeCards(start, end)

[====================] 100%

In [30]:
columns = ['cardId', 'cardType', 'name', 'img', 'regulation', 'setNum', 'setCount', 'rarity', 'dexNum',
            'dexClass', 'height', 'weight', 'dexDesc', 'author', 'desc', 'stage', 'hp', 'pType',
            'ability', 'abilityDesc', 'waza1Cost', 'waza1Name', 'waza1Damage',
            'waza1Desc', 'waza2Cost', 'waza2Name', 'waza2Damage', 'waza2Desc',
            'GXCost', 'GXName', 'GXDamage', 'GXDesc',
            'weakType', 'weakValue', 'resistType', 'resistValue', 'escape', 'spRule']

df = pd.DataFrame(columns=columns)
df

0

In [27]:
cardId = 32388
df.loc[888] = readCard(getContent(cardId))
df

,cardId,cardType,name,img,regulation,setNum,setCount,rarity,dexNum,dexClass,...,GXCost,GXName,GXDamage,GXDesc,weakType,weakValue,resistType,resistValue,escape,spRule
2,32388,pokemon,ウインディBREAK,/assets/images/card_images/legend/XYP/032388_P...,XYP,267,Y-P,,,,...,,,,,,,,,0,BREAK進化する前のウインディの「ワザ・特性・弱点・抵抗力・にげる」を引きつぐ。
888,32388,pokemon,ウインディBREAK,/assets/images/card_images/legend/XYP/032388_P...,XYP,267,Y-P,,,,...,,,,,,,,,0,BREAK進化する前のウインディの「ワザ・特性・弱点・抵抗力・にげる」を引きつぐ。


In [29]:
len(df)

2

In [704]:
import sys

n = 21
for i in range(n):
    j = (i + 1) / n
    sys.stdout.write('\r')
    # the exact output you're looking for:
    sys.stdout.write("[%-20s] %d%%" % ('='*int(20*j), 100*j))
    sys.stdout.flush()

[====================] 100%

In [706]:
print("[%-20s]" % '0')

[0                   ]


In [22]:
'megamark'[:4]

'mega'

In [3]:
print(f"\thahaha")

	hahaha
